In [1]:
######################################################################################
# Authored by Andrew Wan Ju Kang, Ph.D.
# soarhigh0714@gmail.com
# Do not re-distribute.
######################################################################################

In [2]:
import cv2
import base64
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Gemma4 최신 모델인데 영상을 바로 입력으로 줄 수 없나?
# -> 어떤 프레임워크를 통해 쓰는지에 따라 다름
# 1. 허깅페이스 transformers 를 통해 load한다: 가능. messages 구성에 type: video 입력으로 바로 받을 수 있음
# 2. Ollama 방식으로 load한다: 아직 native video 입력 미지원. 프레임 직접 추출 후 이미지 시퀀스로 제공.
# 3. Llama.cpp 방식에서는 아직 native video 입력 미지원. 그래서 비디오에서 프레임 추출 필요 get_video_frames 구현 참고.


# 프레임 단위 추출
def get_video_frames(video_path, max_frames=6):
    video = cv2.VideoCapture(video_path)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total_frames // max_frames)
    
    encoded_frames = []
    for i in range(max_frames):
        video.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        success, frame = video.read()
        if not success: break
            
        # Downscale to 512px to protect VRAM and context limits
        h, w = frame.shape[:2]
        if max(h, w) > 512:
            scale = 512 / max(h, w)
            frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

        _, buffer = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
        b64_str = base64.b64encode(buffer).decode('utf-8')
        
        encoded_frames.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{b64_str}"}
        })
    video.release()
    return encoded_frames

# 넘어짐 샘플 비디오
video_frames = get_video_frames("fall_sample.mp4", max_frames=6)

# ─── 서버 설정 ───
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8181

# ─── LLM 연결: llama-server → OpenAI 호환 API ───
llm = ChatOpenAI(
    model="gemma",                                       # 모델명 (llama-server에서는 무시됨)
    base_url=f"http://{SERVER_HOST}:{SERVER_PORT}/v1",          # 로컬 llama-server
    api_key="not-needed",                                       # 로컬 서버이므로 불필요
    temperature=0.1,
    max_tokens=2048,
)


# 분석 요청 프롬프트
messages = [
    SystemMessage(content="""당신은 각종 산업 현장에서 안전 모니터링을 담당하고 있습니다.
    주어진 데이터에서 낙상 등의 위험 요소가 발견될 경우,
    감지한 위험 요소와 대응책을 한국어로 안내하세요."""),
    HumanMessage(
        content=[
            {"type": "text", "text": "주어진 일련의 비디오 프레임을 분석해 위험 요소를 감지하고 적정 조치를 안내하세요."}
        ] + video_frames
    )
]

print(f"{len(video_frames)}프레임 분석 중... 잠시만 기다려주세요.")
response = llm.invoke(messages)

print("\n=== 현장 모니터링 결과 ===")
print(response.content)

6프레임 분석 중... 잠시만 기다려주세요.

=== 현장 모니터링 결과 ===
제공해주신 비디오 프레임들을 분석한 결과, 다음과 같은 안전 위험 요소와 이에 대한 적정 조치를 안내해 드립니다.

---

### 🚨 감지된 위험 요소

**1. 미끄러운 노면으로 인한 낙상 위험 (Slip and Fall Hazard)**

*   **상황 분석:** 영상은 눈이나 얼음이 덮인 도로 위에서 사람이 개와 함께 걷고 있는 모습을 보여줍니다. 이러한 환경은 지면의 마찰력을 급격히 감소시켜 발이 미끄러지거나 넘어지는 **낙상 사고**의 위험을 매우 높입니다.
*   **주요 위험:** 발목 염좌, 골절, 또는 기타 신체적 부상.

---

### ✅ 적정 대응책 (Safety Countermeasures)

**1. 개인 안전 조치 (Personal Safety Measures)**

*   **보행 속도 조절:** 평소보다 훨씬 느린 속도로 보행하여 발이 노면을 안정적으로 잡을 수 있도록 합니다.
*   **적절한 신발 착용:** 미끄럼 방지 기능이 있는 **미끄럼 방지 기능이 있는 신발(Non-slip footwear)**을 착용하여 접지력을 확보해야 합니다.
*   **균형 유지:** 보폭을 줄이고 몸의 중심을 낮추어 균형을 유지하며 걷습니다.
*   **시선 집중:** 발밑의 노면 상태를 지속적으로 확인하며 걷는 습관을 들입니다.

**2. 환경 및 운영 조치 (Environmental and Operational Measures)**

*   **경고 표지판 확인:** 해당 도로에 미끄러움이나 위험에 대한 경고 표지판이 설치되어 있는지 사전에 확인합니다.
*   **운행 제한:** 기상 조건(눈, 결빙)이 매우 나쁠 경우, 해당 구역의 통행을 일시적으로 제한하거나 운행을 중단해야 합니다.
*   **안전 장비 점검:** 작업자(보행자)가 안전 장비(미끄럼 방지 신발 등)를 올바르게 착용했는지 주기적으로 점검합니다.

---

**요약:** 현재 상황은 **미끄